# NAS - Top-K Beam Selection

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

### **Data Summary**

- **beam_output**  
  - **Shape:** (9638, 8, 32)  
  - **Represents:** Target labels representing beam scores for each sample.  
  - **Example:** For one sample, a portion of the beam scores might look like:  
    ```
    [[1.51e-06, 1.76e-06, 3.01e-06, ...], 
     [1.77e-06, 2.02e-06, 3.37e-06, ...], 
     ...]
    ```  
    Each value is a float (with an imaginary part of zero), indicating the quality of a specific beam pair.

- **coord_input**  
  - **Shape:** (9638, 2)  
  - **Represents:** 2D coordinates associated with each sample (e.g., spatial positions).  
  - **Example:** The first sample might have coordinates similar to:  
    ```
    [748.92, 624.72]
    ```

- **image_input**  
  - **Shape:** (9638, 48, 81, 1)  
  - **Represents:** Grayscale image data where each pixel is an 8-bit unsigned integer.  
  - **Example:** A snippet from the first image might include pixel values such as:  
    ```
    [[[156], [136], [119], ...],
     [[131], [105], [91], ...],
     [[153], [116], [100], ...],
     ...]
    ```

- **lidar_input**  
  - **Shape:** (9638, 20, 200, 10)  
  - **Represents:** LIDAR data formatted as a multi-channel grid (20×200 with 10 channels), encoding spatial features or intensities. This representation indicates that LIDAR data contains spatial information about obstructions, base stations, and the target vehicle, which can be crucial for predicting beamforming paths.
  - **Semantic Meaning of Voxel Values:**  
    - **-2:** Base Station (BS) location  
    - **-1:** Target vehicle (receiver)  
    - **1:** Obstacles (e.g., other vehicles, buildings, pedestrians, trees)  
    - **0:** Empty space (free path for mmWave signals)  
  - **Example:**  
    ```
    [[[0, 0, 0, ..., 0, 0, 0],
      [0, 0, 0, ..., 0, 0, 0],
      ...,
      [-2, -1, 1, ..., 0, 0, 0]],
     ...]
    ```

## 1. Imports

In [1]:
# --------------------------- Standard Libraries ---------------------------- #
import os
import traceback
from IPython.display import clear_output, display, HTML

# -------------------------------- Annotations ------------------------------- #
from typing import List, Optional, Tuple

# ------------------------- Data Processing Libraries ----------------------- #
import numpy as np

# ----------------------- TensorFlow and Keras Modules ---------------------- #
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# --------------------------------- AutoKeras -------------------------------- #
import autokeras as ak

2025-03-13 08:07:05.861746: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-13 08:07:06.037442: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741864026.109695    7473 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741864026.129891    7473 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-13 08:07:06.284543: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## 2. Utility Functions Definitions

### 2.1. GPU

In [2]:
def get_gpu_info():
    """
    Retrieves and prints detailed GPU information including TensorFlow,
    CUDA, cuDNN versions, number of GPUs, and memory details.
    """
    # Display TensorFlow version
    print(f"TensorFlow Version: {tf.__version__}")

    # Check if TensorFlow is built with CUDA support and retrieve build info
    if tf.test.is_built_with_cuda():
        build_info = tf.sysconfig.get_build_info()
        print(f"TensorFlow is built with CUDA support")
        print(f"CUDA Version: {build_info['cuda_version']}")
        print(f"cuDNN Version: {build_info['cudnn_version']}")
    else:
        print("Running on CPU (No CUDA support detected)")

    # Detect available GPUs
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        print(f"\nNumber of GPUs detected: {len(gpus)}")
        print(f"Available GPU(s): {[gpu.name for gpu in gpus]}\n")
        tf.test.gpu_device_name()
    else:
        print("No GPUs found")
        print("Running on CPU")

### 2.2. Folder and Files

In [3]:
def create_run_directory(prefix: str, base_dir: str = "runs") -> str:
    """
    Creates a new directory for storing training logs, checkpoints, and plots.
    The directory name is based on the next available number.

    Args:
        prefix (str): Prefix for the run directory.
        base_dir (str): Base directory for storing training runs. Defaults to "runs".

    Returns:
        str: Path to the created run directory.
    """
    os.makedirs(base_dir, exist_ok=True)  # Ensure the base directory exists

    # Find the next available run number
    existing_dirs = [d for d in os.listdir(base_dir) if d.startswith(prefix) and d[len(prefix):].isdigit()]
    next_run_number = max([int(d[len(prefix):]) for d in existing_dirs] + [0]) + 1
    run_dir = os.path.join(base_dir, f"{prefix}{next_run_number}")
    os.makedirs(run_dir, exist_ok=True)  # Create the run directory

    return run_dir

## 3. Setup and Configuration

### 3.1. GPU Management

In [4]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()

TensorFlow Version: 2.18.0
TensorFlow is built with CUDA support
CUDA Version: 12.5.1
cuDNN Version: 9

Number of GPUs detected: 1
Available GPU(s): ['/physical_device:GPU:0']



I0000 00:00:1741864069.659412    7473 gpu_device.cc:2022] Created device /device:GPU:0 with 2179 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


### 3.2. Random Seed

In [5]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

### 3.3. Run Directory 

In [6]:
# Set to an existing path to resume training
RESUME_TRAINING_PATH = None  # "runs/nas_ak_2" 

RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_ak_")

print(f"Run directory: {RUN_DIR}")

Run directory: runs/nas_ak_5


## 4. Data Loading and Preprocessing

### 4.1. Data Loading

In [7]:
def convert_to_one_hot(y: np.ndarray) -> np.ndarray:
    """
    Converts beam scores into one-hot encoded format where only the highest value 
    in each sample is set to 1, and the rest are 0.

    Args:
        y (np.ndarray): Original beam scores (shape: (N, 8, 32)).

    Returns:
        np.ndarray: One-hot encoded labels (shape: (N, 256)).
    """
    # Flatten last two dimensions (8x32 -> 256)
    y_reshaped = y.reshape(y.shape[0], -1)  # Shape: (N, 256)

    # Find the index of the maximum value for each sample
    max_indices = np.argmax(y_reshaped, axis=1)  # Shape: (N,)

    # Create one-hot encoding
    y_one_hot = np.zeros_like(y_reshaped)  # Shape: (N, 256)
    y_one_hot[np.arange(y.shape[0]), max_indices] = 1  # Set max index to 1

    return y_one_hot

In [8]:
# Define the base directory for data files
DATA_DIR = "./data/s008"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s008_y_train = np.load(beam_output_path)
s008_coord_input = np.load(coord_input_path)
s008_image_input = np.load(image_input_path)
s008_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s008_y_train = s008_y_train.astype(np.float32)
s008_coord_input = s008_coord_input.astype(np.float32)

s008_y_train = convert_to_one_hot(s008_y_train)

# Print the shapes of the loaded data
print(f"y_train shape: {s008_y_train.shape}")
print(f"coord_input shape: {s008_coord_input.shape}")
print(f"image_input shape: {s008_image_input.shape}")
print(f"lidar_input shape: {s008_lidar_input.shape}")

y_train shape: (1960, 256)
coord_input shape: (1960, 2)
image_input shape: (1960, 48, 81, 1)
lidar_input shape: (1960, 20, 200, 10)


/tmp/ipykernel_7473/1741718671.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)


In [9]:
# Define the base directory for data files
DATA_DIR = "./data/s009"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s009_y_train = np.load(beam_output_path)
s009_coord_input = np.load(coord_input_path)
s009_image_input = np.load(image_input_path)
s009_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s009_y_train = s009_y_train.astype(np.float32)
s009_coord_input = s009_coord_input.astype(np.float32)

s009_y_train = convert_to_one_hot(s009_y_train)

# Print the shapes of the loaded data
print(f"y_train shape: {s009_y_train.shape}")
print(f"coord_input shape: {s009_coord_input.shape}")
print(f"image_input shape: {s009_image_input.shape}")
print(f"lidar_input shape: {s009_lidar_input.shape}")

y_train shape: (9638, 256)
coord_input shape: (9638, 2)
image_input shape: (9638, 48, 81, 1)
lidar_input shape: (9638, 20, 200, 10)


/tmp/ipykernel_7473/177270057.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y_train = s009_y_train.astype(np.float32)


In [10]:
# ----------------------- Subsample dataset for testing ---------------------- #
# SAMPLE_SIZE = 50  # Use a subset of x samples for testing
# s008_y_train = s008_y_train[:SAMPLE_SIZE]
# s008_coord_input = s008_coord_input[:SAMPLE_SIZE]
# s008_image_input = s008_image_input[:SAMPLE_SIZE]
# s008_lidar_input = s008_lidar_input[:SAMPLE_SIZE]

# s009_y_train = s009_y_train[:SAMPLE_SIZE]
# s009_coord_input = s009_coord_input[:SAMPLE_SIZE]
# s009_image_input = s009_image_input[:SAMPLE_SIZE]
# s009_lidar_input = s009_lidar_input[:SAMPLE_SIZE]

# # Print the shapes of the loaded data
# print(f"y_train s008 shape: {s008_y_train.shape}")
# print(f"coord_input s008 shape: {s008_coord_input.shape}")
# print(f"image_input s008 shape: {s008_image_input.shape}")
# print(f"lidar_input s008 shape: {s008_lidar_input.shape}")

# print(f"y_train s009 shape: {s009_y_train.shape}")
# print(f"coord_input s009 shape: {s009_coord_input.shape}")
# print(f"image_input s009 shape: {s009_image_input.shape}")
# print(f"lidar_input s009 shape: {s009_lidar_input.shape}")

## 5. NAS Study Setup AK

In [ ]:
def build_auto_model(num_trials: int = 200) -> ak.AutoModel:
    """Build and return an AutoKeras model that integrates multiple inputs for
    predicting beam scores.

    The model accepts three inputs:
      - image_input: Grayscale image with shape (48, 81, 1)
      - coord_input: 2D coordinates with shape (2,)
      - lidar_input: LIDAR data with shape (20, 200, 10)

    It outputs a regression prediction with shape (8, 32) corresponding to beam scores.

    Args:
        num_trials (int, optional): The number of trials for AutoKeras to search for the best model. Defaults to 200.

    Returns:
        ak.AutoModel: A compiled AutoKeras AutoModel configured for regression.
    """
    # Define each input node
    image_input = ak.ImageInput(shape=(48, 81, 1), name="image_input")
    coord_input = ak.Input(shape=(2,), name="coord_input")
    lidar_input = ak.Input(shape=(20, 200, 10), name="lidar_input")

    # Define the classification head with softmax,
    # just like regression but with a normalization, that is the softmax
    # https://autokeras.com/block/#classificationhead
    output = ak.ClassificationHead(
        num_classes=256,
        multi_label=False,
        metrics=["accuracy"],
    )

    automodel = ak.AutoModel(
        inputs=[image_input, coord_input, lidar_input],
        outputs=output,
        max_trials=num_trials,
        overwrite=False,  # Setting overwrite=False enables resume training
        directory=os.path.join(RUN_DIR, "auto_model"),
    )
    return automodel

## 6. Do the NAS

### 6.1. Code Health Check

In [ ]:
# ------------------------------- Log Resources ------------------------------ #
#! Remove this block if you don't want to use it
# try:
#     from utils import log_resources
#     LOG_DIR = os.path.join(RUN_DIR, "logs")
#     os.makedirs(LOG_DIR, exist_ok=True)
    
#     log_resources.log_resources(
#         log_dir=LOG_DIR,
#         interval=5,
#         cpu=False,
#         ram=True,
#         gpu=True,
#         cuda=False,
#         tensorflow=False,
#     )
# except Exception as e:
#     print("[ERROR] Failed to log resources!")
#     print(e)
#     pass

In [ ]:
# ---------------------------- Kernel life monitor --------------------------- #
#! Remove this block if you don't want to use it
try:
    pid = os.getpid()
    display(HTML(f'Call the monitor script: <span style="color: orange;">python _monitor_kernel_life.py --pid {pid}</span>'))
except Exception as e:
    print("[ERROR] Kernel monitoring failed!")
    print(e)
    pass

### 6.2. Main

In [ ]:
# ----------------------------- Hyperparameters ------------------------------ #
NUM_TRIALS = 200
EPOCHS = None
BATCH_SIZE = 32

In [ ]:
model_path = os.path.join(RUN_DIR, "models")
os.makedirs(model_path, exist_ok=True)

# -------------------------- Build and Train Model -------------------------- #
automodel = build_auto_model(num_trials=NUM_TRIALS)

automodel.fit(
    x=[s008_image_input, s008_coord_input, s008_lidar_input],
    y=s008_y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=None,
    validation_data=(
        [s009_image_input, s009_coord_input, s009_lidar_input],
        s009_y_train
    ),
    # validation_split=0.1, # Overwritten by validation_data
    verbose=2,
)

# -------------------------- Save the trained model -------------------------- #
best_model = automodel.export_model()
best_model.save(os.path.join(model_path, "best_model.keras"))

best_model.summary()

### 6.3. Email Notification

In [ ]:
# ----------------------- Email api to notify the user ----------------------- #
#! Remove this block if you don't want to send an email
try:
    from utils.email_api import send_email

    # Message that the NAS Beam Selection study is complete
    email_subject = "NAS Beam Selection Study Complete"
    email_body = """
    <html>
        <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333; background-color: #f9f9f9; padding: 20px;">
        <div style="max-width: 600px; margin: auto; background: #fff; padding: 20px; border: 1px solid #ddd; border-radius: 8px;">
            <h2 style="color: #0056b3; text-align: center;">🎉 NAS Beam Selection Study Complete</h2>
            <p style="font-size: 16px; color: #444;">
            <strong>Dear User,</strong>
            </p>
            <p style="font-size: 18px; color: #333;">
            Your NAS Beam Selection study has successfully completed!
            </p>
            <p style="text-align: center; font-size: 16px;">
            <strong style="color: #28a745;">✔️ Study Status:</strong> <span style="color: #0056b3;">Completed</span>
            </p>
            <footer style="margin-top: 20px; text-align: center; font-size: 14px; color: #888;">
            <p>Best regards,</p>
            <p><strong>The Optimization Team</strong></p>
            </footer>
        </div>
        </body>
    </html>
    """
    send_email(
        subject=email_subject,
        body=email_body,
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        text_type="html",
    )
except Exception as e:
    print(f"[ERROR] Failed to send email: {e}")
    traceback.print_exc()
    pass

## 7. Loading the Model & Calculating Accuracy

In [ ]:
# Define path to best trained model
model_path = os.path.join(RUN_DIR, "models", "best_model.keras")

# Load the trained model
loaded_model = tf.keras.models.load_model(model_path)

# ---------------------------- Evaluate Accuracy ---------------------------- #
def evaluate_model(model, x_test, y_test):
    """
    Evaluates the trained model on a given test dataset and prints accuracy.
    
    Args:
        model (tf.keras.Model): The trained model.
        x_test (List[np.ndarray]): List of test inputs.
        y_test (np.ndarray): Ground truth labels.
    """
    results = model.evaluate(x_test, y_test, verbose=2)
    
    # Extract accuracy if available
    loss = results[0]
    accuracy = results[1] if len(results) > 1 else None
    
    print(f"Test Loss: {loss:.6f}")
    if accuracy is not None:
        print(f"Test Accuracy: {accuracy:.6%}")
    else:
        print("[WARNING] Model does not output accuracy. Ensure correct classification setup.")

# ---------------------------- Run Evaluation ---------------------------- #
evaluate_model(
    loaded_model, 
    [s009_image_input, s009_coord_input, s009_lidar_input], 
    s009_y_train
)

I0000 00:00:1741864089.843783    7473 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2179 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
/home/matheus/anaconda3/envs/tf-ak/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 445 variables whereas the saved optimizer has 1 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/home/matheus/anaconda3/envs/tf-ak/lib/python3.11/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['input_layer', 'input_layer_1', 'input_layer_2']. Received: the structure of inputs=('*', '*', '*')
  warnings.warn(
I0000 00:00:1741864096.190819    8171 service.cc:148] XLA service 0x7ee314002860 initialized for platform CUDA (this does not guarantee that XLA will be used). D

302/302 - 16s - 54ms/step - accuracy: 0.0064 - loss: 4.7085
Test Loss: 4.708457
Test Accuracy: 0.643287%
